# Data cleaning demo

This notebook runs the same Python curation functions as the main workflow. The bundled records and molecular-weight lookup require no API key or PDF access.

Install from the repository root with `python -m pip install -e ".[curation,datasets,notebook]"`.

## Inputs and setup

Run the cells in order. The following paths are relative to `Demo/01_data_cleaning/`; all required files are included.

| Input | Location | Contents |
| --- | --- | --- |
| Raw extraction records, CSV | `input/mof_extraction.csv` | Original extraction fields plus `has_main_document` and `has_supporting_document` as `True`/`False` flags. |
| Molecular-weight lookup, CSV | `input/linker_molecular_weights.csv` | Two headerless columns: linker name and molecular weight. |
| Linker prime corrections, JSON | `../../data/lookups/linker_prime_corrections.json` | Included exact DOI/name pairs shared with the main curation workflow. |
| Supplied cleaned reference, CSV | `reference/positive_stage6_supplied.csv` | A comparison table; it is not used as a curation input. |

To test another raw extraction, save a CSV with the same schema, select it in `config.json`, and change `run_demo(check=True)` below to `run_demo(check=False)`. The availability flags must reflect whether each source document was present. For extraction outputs that retain document-path columns, use [the main curation notebook](../../notebooks/05_data_curation.ipynb).

The demo writes stage tables and a before/after preview under `outputs/`. With the supplied inputs, `check=True` compares results against `expected/`. Continue with [JSON preparation](../02_json_preparation/demo.ipynb) after cleaning.

Implementation: [demo runner](run_demo.py) and [curation stages](../../src/mofinder/curation/pipeline.py). See the [source-to-code guide](../../docs/source_to_code.md) for the original workflow stages and their corresponding functions.


In [ ]:
from pathlib import Path
import runpy
import pandas as pd

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "mofinder").is_dir():
        REPO = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from within the MOFinder repository.")

DEMO = REPO / "Demo" / "01_data_cleaning"


## 1. Inspect the extraction records

Document-availability flags retain the original filter without local file paths. The full records are available in `input/mof_extraction.csv`.

In [ ]:
raw = pd.read_csv(DEMO / "input" / "mof_extraction.csv")
raw[["doi", "mof_name", "metal_1", "linker_1", "time_h", "time_text"]].head(12)


## 2. Run curation through stage 6

The script saves intermediate tables and compares the final result with the bundled expected output.

In [ ]:
run_demo = runpy.run_path(str(DEMO / "run_demo.py"))["run"]
summary = run_demo(check=True)
pd.DataFrame(summary["stages"])


## 3. Inspect normalized conditions and durations

In [ ]:
cleaned = pd.read_csv(DEMO / "outputs" / "cleaned_preview.csv")
cleaned.head(12)


The cleaned stage 6 table can be passed to the companion JSON preparation demo. The separate `reference/` table is the supplied full-corpus cleaned slice; frequency and outlier filters are computed on the demo cohort when regenerating `expected/`.